# 🌍 IMF Global Fossil Fuel Subsidies — Comprehensive EDA

[![Kaggle Dataset](https://img.shields.io/badge/Kaggle-Dataset-blue?logo=kaggle)](https://www.kaggle.com/datasets/zkskhurram)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-green?logo=python)](https://python.org)

---

## 📝 About This Notebook

This notebook performs a **professional Exploratory Data Analysis (EDA)** on the IMF Fossil Fuel Subsidies dataset covering **168 countries** from **2015 to 2030**.

### Table of Contents
1. [Setup & Configuration](#1)
2. [Data Loading & First Look](#2)
3. [Data Cleaning & Preparation](#3)
4. [Statistical Summary](#4)
5. [Global Subsidy Trends Over Time](#5)
6. [Top Subsidizing Countries](#6)
7. [Explicit vs Implicit Subsidies](#7)
8. [Fuel Type Breakdown](#8)
9. [Externality Analysis](#9)
10. [Regional & Continental Analysis](#10)
11. [Correlation & Distribution Analysis](#11)
12. [Key Insights & Conclusions](#12)

---

**Author:** Khurram Shahzad \
**Mentor:** Dr. Aammar Tufail \
**Data Source:** [IMF Climate Change Indicators](https://climatedata.imf.org/pages/mitigation/#mi3)  \
**Environment:** `kag_go` (Conda)

<a id='1'></a>
## 1. ⚙️ Setup & Configuration

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import json
import os

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# ---- Matplotlib / Seaborn Style ----
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'figure.dpi': 100,
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'font.family': 'sans-serif',
})

# ---- Plotly Template ----
PLOTLY_TEMPLATE = 'plotly_white'
COLOR_PALETTE = px.colors.qualitative.Bold

# ---- Pandas Display ----
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 60)

print('✅ All libraries loaded successfully!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   plotly  : {px.__version__ if hasattr(px, "__version__") else "5.x"}')

<a id='2'></a>
## 2. 📥 Data Loading & First Look

In [ ]:
# ============================================================
# LOAD DATA — Auto-detect Kaggle vs Local path
# ============================================================
KAGGLE_PATH = '/kaggle/input/datasets/zkskhurram/imf-global-fossil-fuel-subsidies-20152030/IMF_FFS.csv'

csv_path = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH
df = pd.read_csv(csv_path)

print(f'✅ Dataset loaded from: {csv_path}')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Column info
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
print('\n🔍 Missing Values Summary:')
print(missing_df[missing_df['Missing'] > 0].to_string() if missing_df['Missing'].sum() > 0 else '   ✅ No missing values (except COMMENT_OBS which is entirely null)')
print(f'\n   Total null cells: {missing.sum():,} out of {df.shape[0] * df.shape[1]:,}')

In [ ]:
# Quick uniqueness check for key columns
key_cols = ['REF_AREA', 'REF_AREA_LABEL', 'INDICATOR_LABEL', 'UNIT_MEASURE_LABEL', 'TIME_PERIOD']
print('📊 Unique Value Counts:')
print('─' * 50)
for col in key_cols:
    print(f'   {col:30s} → {df[col].nunique():>5,}')
print('─' * 50)
print(f'   Year Range: {df["TIME_PERIOD"].min()} – {df["TIME_PERIOD"].max()}')

<a id='3'></a>
## 3. 🧹 Data Cleaning & Preparation

In [ ]:
# ============================================================
# DROP REDUNDANT COLUMNS & CREATE ANALYSIS-READY VIEWS
# ============================================================

# Columns that are constant or administrative
drop_cols = [
    'STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'FREQ', 'FREQ_LABEL',
    'SEX', 'SEX_LABEL', 'AGE', 'AGE_LABEL', 'URBANISATION', 'URBANISATION_LABEL',
    'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_1_LABEL',
    'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_2_LABEL',
    'COMP_BREAKDOWN_3', 'COMP_BREAKDOWN_3_LABEL',
    'DATABASE_ID', 'DATABASE_ID_LABEL', 'UNIT_MULT', 'UNIT_MULT_LABEL',
    'UNIT_TYPE', 'UNIT_TYPE_LABEL', 'TIME_FORMAT', 'TIME_FORMAT_LABEL',
    'COMMENT_OBS', 'OBS_STATUS', 'OBS_STATUS_LABEL', 'OBS_CONF', 'OBS_CONF_LABEL'
]

df_clean = df.drop(columns=drop_cols)

# Rename for clarity
df_clean = df_clean.rename(columns={
    'REF_AREA': 'country_code',
    'REF_AREA_LABEL': 'country',
    'INDICATOR': 'indicator_code',
    'INDICATOR_LABEL': 'indicator',
    'UNIT_MEASURE': 'unit_code',
    'UNIT_MEASURE_LABEL': 'unit',
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'value'
})

# Create separate DataFrames by unit
df_gdp = df_clean[df_clean['unit_code'] == 'PT_GDP'].copy()
df_usd = df_clean[df_clean['unit_code'] == 'USD_K_2021'].copy()

# Convert USD to billions for readability
df_usd['value_bn'] = df_usd['value'] / 1e9

print(f'✅ Cleaned DataFrame: {df_clean.shape}')
print(f'   GDP % subset : {df_gdp.shape[0]:,} rows')
print(f'   USD subset   : {df_usd.shape[0]:,} rows')
df_clean.head()

In [ ]:
# ============================================================
# ADD REGION MAPPING (Continental groupings)
# ============================================================

region_map = {
    # East Asia & Pacific
    'AUS':'East Asia & Pacific','BRN':'East Asia & Pacific','KHM':'East Asia & Pacific',
    'CHN':'East Asia & Pacific','FJI':'East Asia & Pacific','IDN':'East Asia & Pacific',
    'JPN':'East Asia & Pacific','KIR':'East Asia & Pacific','KOR':'East Asia & Pacific',
    'LAO':'East Asia & Pacific','MYS':'East Asia & Pacific','MNG':'East Asia & Pacific',
    'MMR':'East Asia & Pacific','NZL':'East Asia & Pacific','PNG':'East Asia & Pacific',
    'PHL':'East Asia & Pacific','SGP':'East Asia & Pacific','SLB':'East Asia & Pacific',
    'THA':'East Asia & Pacific','TON':'East Asia & Pacific','VNM':'East Asia & Pacific',
    # Europe & Central Asia
    'ALB':'Europe & Central Asia','ARM':'Europe & Central Asia','AUT':'Europe & Central Asia',
    'AZE':'Europe & Central Asia','BLR':'Europe & Central Asia','BEL':'Europe & Central Asia',
    'BIH':'Europe & Central Asia','BGR':'Europe & Central Asia','HRV':'Europe & Central Asia',
    'CYP':'Europe & Central Asia','CZE':'Europe & Central Asia','DNK':'Europe & Central Asia',
    'EST':'Europe & Central Asia','FIN':'Europe & Central Asia','FRA':'Europe & Central Asia',
    'GEO':'Europe & Central Asia','DEU':'Europe & Central Asia','GRC':'Europe & Central Asia',
    'HUN':'Europe & Central Asia','ISL':'Europe & Central Asia','IRL':'Europe & Central Asia',
    'ITA':'Europe & Central Asia','KAZ':'Europe & Central Asia','KGZ':'Europe & Central Asia',
    'LVA':'Europe & Central Asia','LTU':'Europe & Central Asia','LUX':'Europe & Central Asia',
    'MKD':'Europe & Central Asia','MDA':'Europe & Central Asia','NLD':'Europe & Central Asia',
    'NOR':'Europe & Central Asia','POL':'Europe & Central Asia','PRT':'Europe & Central Asia',
    'ROU':'Europe & Central Asia','RUS':'Europe & Central Asia','SRB':'Europe & Central Asia',
    'SVK':'Europe & Central Asia','SVN':'Europe & Central Asia','ESP':'Europe & Central Asia',
    'SWE':'Europe & Central Asia','CHE':'Europe & Central Asia','TJK':'Europe & Central Asia',
    'TUR':'Europe & Central Asia','TKM':'Europe & Central Asia','UKR':'Europe & Central Asia',
    'GBR':'Europe & Central Asia','UZB':'Europe & Central Asia','MLT':'Europe & Central Asia',
    # Latin America & Caribbean
    'ARG':'Latin America & Caribbean','BHS':'Latin America & Caribbean','BRB':'Latin America & Caribbean',
    'BLZ':'Latin America & Caribbean','BOL':'Latin America & Caribbean','BRA':'Latin America & Caribbean',
    'CHL':'Latin America & Caribbean','COL':'Latin America & Caribbean','CRI':'Latin America & Caribbean',
    'DOM':'Latin America & Caribbean','ECU':'Latin America & Caribbean','SLV':'Latin America & Caribbean',
    'GTM':'Latin America & Caribbean','GUY':'Latin America & Caribbean','HTI':'Latin America & Caribbean',
    'HND':'Latin America & Caribbean','JAM':'Latin America & Caribbean','MEX':'Latin America & Caribbean',
    'NIC':'Latin America & Caribbean','PAN':'Latin America & Caribbean','PRY':'Latin America & Caribbean',
    'PER':'Latin America & Caribbean','LCA':'Latin America & Caribbean','SUR':'Latin America & Caribbean',
    'TTO':'Latin America & Caribbean','URY':'Latin America & Caribbean','VEN':'Latin America & Caribbean',
    # Middle East & North Africa
    'DZA':'Middle East & North Africa','BHR':'Middle East & North Africa','DJI':'Middle East & North Africa',
    'EGY':'Middle East & North Africa','IRN':'Middle East & North Africa','IRQ':'Middle East & North Africa',
    'ISR':'Middle East & North Africa','JOR':'Middle East & North Africa','KWT':'Middle East & North Africa',
    'LBN':'Middle East & North Africa','LBY':'Middle East & North Africa','MAR':'Middle East & North Africa',
    'OMN':'Middle East & North Africa','QAT':'Middle East & North Africa','SAU':'Middle East & North Africa',
    'TUN':'Middle East & North Africa','ARE':'Middle East & North Africa','YEM':'Middle East & North Africa',
    'MRT':'Middle East & North Africa',
    # North America
    'CAN':'North America','USA':'North America',
    # South Asia
    'AFG':'South Asia','BGD':'South Asia','BTN':'South Asia','IND':'South Asia',
    'MDV':'South Asia','NPL':'South Asia','PAK':'South Asia','LKA':'South Asia',
    # Sub-Saharan Africa
    'AGO':'Sub-Saharan Africa','BEN':'Sub-Saharan Africa','BWA':'Sub-Saharan Africa',
    'BFA':'Sub-Saharan Africa','BDI':'Sub-Saharan Africa','CPV':'Sub-Saharan Africa',
    'CMR':'Sub-Saharan Africa','CAF':'Sub-Saharan Africa','TCD':'Sub-Saharan Africa',
    'COM':'Sub-Saharan Africa','COD':'Sub-Saharan Africa','COG':'Sub-Saharan Africa',
    'CIV':'Sub-Saharan Africa','GNQ':'Sub-Saharan Africa','ETH':'Sub-Saharan Africa',
    'GAB':'Sub-Saharan Africa','GMB':'Sub-Saharan Africa','GHA':'Sub-Saharan Africa',
    'GIN':'Sub-Saharan Africa','GNB':'Sub-Saharan Africa','KEN':'Sub-Saharan Africa',
    'LSO':'Sub-Saharan Africa','LBR':'Sub-Saharan Africa','MDG':'Sub-Saharan Africa',
    'MWI':'Sub-Saharan Africa','MLI':'Sub-Saharan Africa','MUS':'Sub-Saharan Africa',
    'MOZ':'Sub-Saharan Africa','NAM':'Sub-Saharan Africa','NER':'Sub-Saharan Africa',
    'NGA':'Sub-Saharan Africa','RWA':'Sub-Saharan Africa','STP':'Sub-Saharan Africa',
    'SEN':'Sub-Saharan Africa','SYC':'Sub-Saharan Africa','SLE':'Sub-Saharan Africa',
    'ZAF':'Sub-Saharan Africa','SDN':'Sub-Saharan Africa','TZA':'Sub-Saharan Africa',
    'TGO':'Sub-Saharan Africa','UGA':'Sub-Saharan Africa','ZMB':'Sub-Saharan Africa',
    'ZWE':'Sub-Saharan Africa',
}

df_clean['region'] = df_clean['country_code'].map(region_map).fillna('Other')
df_gdp['region'] = df_gdp['country_code'].map(region_map).fillna('Other')
df_usd['region'] = df_usd['country_code'].map(region_map).fillna('Other')

print('✅ Region mapping applied')
print(df_clean['region'].value_counts())

<a id='4'></a>
## 4. 📊 Statistical Summary

In [ ]:
# ============================================================
# DATASET OVERVIEW CARD
# ============================================================
total_indicator = 'Fossil Fuel Subsidies - Total Implicit and Explicit'

total_usd_2025 = df_usd[
    (df_usd['indicator'] == total_indicator) & (df_usd['year'] == 2025)
]['value'].sum()

avg_gdp_2025 = df_gdp[
    (df_gdp['indicator'] == total_indicator) & (df_gdp['year'] == 2025)
]['value'].mean()

max_gdp_row = df_gdp[
    (df_gdp['indicator'] == total_indicator) & (df_gdp['year'] == 2025)
].nlargest(1, 'value').iloc[0]

print('=' * 60)
print('       🌍 GLOBAL FOSSIL FUEL SUBSIDIES — 2025 SNAPSHOT')
print('=' * 60)
print(f'  Global Total (USD 2021)  : ${total_usd_2025/1e12:,.2f} Trillion')
print(f'  Average (% of GDP)       : {avg_gdp_2025:.2f}%')
print(f'  Highest (% of GDP)       : {max_gdp_row["country"]} — {max_gdp_row["value"]:.2f}%')
print(f'  Countries Covered        : {df_clean["country"].nunique()}')
print(f'  Year Range               : {df_clean["year"].min()} – {df_clean["year"].max()}')
print(f'  Total Observations       : {len(df_clean):,}')
print('=' * 60)

In [ ]:
# Statistical describe for GDP % values
total_gdp_stats = df_gdp[df_gdp['indicator'] == total_indicator]
print('\n📊 Descriptive Statistics — Total Subsidies (% of GDP):')
total_gdp_stats.groupby('year')['value'].describe().round(2)

<a id='5'></a>
## 5. 📈 Global Subsidy Trends Over Time

In [ ]:
# ============================================================
# GLOBAL TOTAL SUBSIDIES TREND (USD Trillions)
# ============================================================
trend_data = df_usd[
    df_usd['indicator'] == total_indicator
].groupby('year')['value'].sum().reset_index()
trend_data['value_trillion'] = trend_data['value'] / 1e12

fig = px.bar(
    trend_data, x='year', y='value_trillion',
    title='<b>🌍 Global Fossil Fuel Subsidies — Total (2015–2030)</b>',
    labels={'year': 'Year', 'value_trillion': 'Total Subsidies (USD Trillions)'},
    color='value_trillion',
    color_continuous_scale='Reds',
    template=PLOTLY_TEMPLATE,
    text=trend_data['value_trillion'].round(2)
)
fig.update_traces(textposition='outside', textfont_size=11)
fig.update_layout(
    height=500, width=900,
    xaxis=dict(dtick=1),
    coloraxis_showscale=False,
    yaxis_title='Trillions (USD, 2021 constant)',
)
fig.show()

In [ ]:
# ============================================================
# AVERAGE % GDP TREND ACROSS ALL COUNTRIES
# ============================================================
avg_gdp_trend = df_gdp[
    df_gdp['indicator'] == total_indicator
].groupby('year')['value'].agg(['mean', 'median']).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=avg_gdp_trend['year'], y=avg_gdp_trend['mean'],
    mode='lines+markers', name='Mean',
    line=dict(width=3, color='#E74C3C'), marker=dict(size=8)
))
fig.add_trace(go.Scatter(
    x=avg_gdp_trend['year'], y=avg_gdp_trend['median'],
    mode='lines+markers', name='Median',
    line=dict(width=3, dash='dash', color='#3498DB'), marker=dict(size=8)
))
fig.update_layout(
    title='<b>📉 Average Fossil Fuel Subsidies (% of GDP) — Global Trend</b>',
    xaxis_title='Year', yaxis_title='Subsidies (% of GDP)',
    template=PLOTLY_TEMPLATE, height=450, width=900,
    xaxis=dict(dtick=1),
    legend=dict(x=0.02, y=0.98)
)
fig.show()

<a id='6'></a>
## 6. 🏆 Top Subsidizing Countries

In [ ]:
# ============================================================
# TOP 20 COUNTRIES BY TOTAL SUBSIDIES (USD) — 2025
# ============================================================
top20_usd = df_usd[
    (df_usd['indicator'] == total_indicator) & (df_usd['year'] == 2025)
].nlargest(20, 'value').copy()
top20_usd['value_bn'] = top20_usd['value'] / 1e9

fig = px.bar(
    top20_usd, x='value_bn', y='country', orientation='h',
    title='<b>🏆 Top 20 Countries — Total Fossil Fuel Subsidies (2025, USD Billions)</b>',
    labels={'value_bn': 'Subsidies (USD Billions)', 'country': ''},
    color='value_bn',
    color_continuous_scale='YlOrRd',
    template=PLOTLY_TEMPLATE,
    text=top20_usd['value_bn'].round(1)
)
fig.update_traces(textposition='outside', textfont_size=11)
fig.update_layout(
    height=700, width=900,
    yaxis=dict(categoryorder='total ascending'),
    coloraxis_showscale=False
)
fig.show()

In [ ]:
# ============================================================
# TOP 20 COUNTRIES BY % GDP — 2025
# ============================================================
top20_gdp = df_gdp[
    (df_gdp['indicator'] == total_indicator) & (df_gdp['year'] == 2025)
].nlargest(20, 'value').copy()

fig = px.bar(
    top20_gdp, x='value', y='country', orientation='h',
    title='<b>💰 Top 20 Countries — Subsidies as % of GDP (2025)</b>',
    labels={'value': 'Subsidies (% of GDP)', 'country': ''},
    color='value',
    color_continuous_scale='Plasma',
    template=PLOTLY_TEMPLATE,
    text=top20_gdp['value'].round(1)
)
fig.update_traces(textposition='outside', textfont_size=11)
fig.update_layout(
    height=700, width=900,
    yaxis=dict(categoryorder='total ascending'),
    coloraxis_showscale=False
)
fig.show()

In [ ]:
# ============================================================
# WORLD MAP — CHOROPLETH (% GDP, 2025)
# ============================================================
map_data = df_gdp[
    (df_gdp['indicator'] == total_indicator) & (df_gdp['year'] == 2025)
].copy()

fig = px.choropleth(
    map_data,
    locations='country_code',
    color='value',
    hover_name='country',
    hover_data={'value': ':.2f', 'country_code': False},
    color_continuous_scale='YlOrRd',
    title='<b>🗺️ Global Fossil Fuel Subsidies — % of GDP (2025)</b>',
    labels={'value': '% of GDP'},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(height=550, width=1000, geo=dict(showframe=False))
fig.show()

<a id='7'></a>
## 7. ⚖️ Explicit vs Implicit Subsidies

In [ ]:
# ============================================================
# GLOBAL EXPLICIT vs IMPLICIT — STACKED TREND
# ============================================================
exp_imp_indicators = [
    'Explicit Fossil Fuel Subsidies - Total',
    'Implicit Fossil Fuel Subsidies - Total'
]

exp_imp_trend = df_usd[
    df_usd['indicator'].isin(exp_imp_indicators)
].groupby(['year', 'indicator'])['value'].sum().reset_index()
exp_imp_trend['value_bn'] = exp_imp_trend['value'] / 1e9
exp_imp_trend['type'] = exp_imp_trend['indicator'].apply(
    lambda x: 'Explicit' if 'Explicit' in x else 'Implicit'
)

fig = px.area(
    exp_imp_trend, x='year', y='value_bn', color='type',
    title='<b>⚖️ Explicit vs Implicit Subsidies — Global Trend (USD Billions)</b>',
    labels={'value_bn': 'USD Billions', 'year': 'Year', 'type': 'Subsidy Type'},
    color_discrete_map={'Explicit': '#E74C3C', 'Implicit': '#3498DB'},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(height=500, width=900, xaxis=dict(dtick=1))
fig.show()

In [ ]:
# ============================================================
# EXPLICIT vs IMPLICIT — PIE CHART (2025)
# ============================================================
exp_imp_2025 = df_usd[
    (df_usd['indicator'].isin(exp_imp_indicators)) & (df_usd['year'] == 2025)
].groupby('indicator')['value'].sum().reset_index()
exp_imp_2025['type'] = exp_imp_2025['indicator'].apply(
    lambda x: 'Explicit' if 'Explicit' in x else 'Implicit'
)
exp_imp_2025['value_tn'] = exp_imp_2025['value'] / 1e12

fig = px.pie(
    exp_imp_2025, values='value', names='type',
    title='<b>🥧 Explicit vs Implicit Share — 2025</b>',
    color='type',
    color_discrete_map={'Explicit': '#E74C3C', 'Implicit': '#3498DB'},
    hole=0.45, template=PLOTLY_TEMPLATE
)
fig.update_traces(textinfo='label+percent+value', textfont_size=13,
                  texttemplate='%{label}<br>%{percent}<br>$%{value:,.0f}')
fig.update_layout(height=450, width=600)
fig.show()

<a id='8'></a>
## 8. ⛽ Fuel Type Breakdown

In [ ]:
# ============================================================
# TOTAL SUBSIDIES BY FUEL TYPE — GLOBAL TREND
# ============================================================
fuel_indicators = [
    'Fossil Fuel Subsidies - Total Implicit and Explicit - Coal',
    'Fossil Fuel Subsidies - Total Implicit and Explicit -  Natural Gas',
    'Fossil Fuel Subsidies - Total Implicit and Explicit - Petroleum',
    'Fossil Fuel Subsidies - Total Implicit and Explicit - Electricity'
]

fuel_trend = df_usd[
    df_usd['indicator'].isin(fuel_indicators)
].groupby(['year', 'indicator'])['value'].sum().reset_index()
fuel_trend['value_bn'] = fuel_trend['value'] / 1e9
fuel_trend['fuel'] = fuel_trend['indicator'].str.extract(r'- (Coal|Natural Gas|Petroleum|Electricity)')[0]

fig = px.bar(
    fuel_trend, x='year', y='value_bn', color='fuel',
    title='<b>⛽ Global Subsidies by Fuel Type (USD Billions)</b>',
    labels={'value_bn': 'USD Billions', 'year': 'Year', 'fuel': 'Fuel Type'},
    barmode='stack',
    color_discrete_map={
        'Coal': '#2C3E50', 'Natural Gas': '#E67E22',
        'Petroleum': '#8E44AD', 'Electricity': '#F1C40F'
    },
    template=PLOTLY_TEMPLATE
)
fig.update_layout(height=500, width=900, xaxis=dict(dtick=1))
fig.show()

In [ ]:
# ============================================================
# FUEL TYPE SHARE — SUNBURST (2025)
# ============================================================
fuel_2025 = df_usd[
    (df_usd['indicator'].isin(fuel_indicators)) & (df_usd['year'] == 2025)
].copy()
fuel_2025['fuel'] = fuel_2025['indicator'].str.extract(r'- (Coal|Natural Gas|Petroleum|Electricity)')[0]

fuel_by_region = fuel_2025.groupby(['region', 'fuel'])['value'].sum().reset_index()
fuel_by_region['value_bn'] = fuel_by_region['value'] / 1e9

fig = px.sunburst(
    fuel_by_region, path=['region', 'fuel'], values='value_bn',
    title='<b>☀️ Subsidy Distribution — Region × Fuel Type (2025)</b>',
    color='fuel',
    color_discrete_map={
        'Coal': '#2C3E50', 'Natural Gas': '#E67E22',
        'Petroleum': '#8E44AD', 'Electricity': '#F1C40F'
    },
    template=PLOTLY_TEMPLATE
)
fig.update_layout(height=600, width=700)
fig.show()

<a id='9'></a>
## 9. 🌫️ Externality Analysis (Implicit Subsidy Breakdown)

In [ ]:
# ============================================================
# IMPLICIT SUBSIDIES BY EXTERNALITY TYPE — 2025
# ============================================================
externality_indicators = [
    'Implicit Fossil Fuel Subsidies - Global Warming',
    'Implicit Fossil Fuel Subsidies - Local Air Pollution',
    'Implicit Fossil Fuel Subsidies - Congestion',
    'Implicit Fossil Fuel Subsidies - Road damage',
    'Implicit Fossil Fuel Subsidies - Accidents',
    'Implicit Fossil Fuel Subsidies - Foregone VAT'
]

ext_2025 = df_usd[
    (df_usd['indicator'].isin(externality_indicators)) & (df_usd['year'] == 2025)
].groupby('indicator')['value'].sum().reset_index()
ext_2025['externality'] = ext_2025['indicator'].str.replace('Implicit Fossil Fuel Subsidies - ', '', regex=False)
ext_2025['value_bn'] = ext_2025['value'] / 1e9
ext_2025 = ext_2025.sort_values('value_bn', ascending=True)

fig = px.bar(
    ext_2025, x='value_bn', y='externality', orientation='h',
    title='<b>🌫️ Implicit Subsidies by Externality Type — 2025 (USD Billions)</b>',
    labels={'value_bn': 'USD Billions', 'externality': ''},
    color='externality',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template=PLOTLY_TEMPLATE,
    text=ext_2025['value_bn'].round(1)
)
fig.update_traces(textposition='outside', textfont_size=12)
fig.update_layout(height=450, width=900, showlegend=False)
fig.show()

In [ ]:
# ============================================================
# EXTERNALITY TRENDS OVER TIME
# ============================================================
ext_trend = df_usd[
    df_usd['indicator'].isin(externality_indicators)
].groupby(['year', 'indicator'])['value'].sum().reset_index()
ext_trend['externality'] = ext_trend['indicator'].str.replace('Implicit Fossil Fuel Subsidies - ', '', regex=False)
ext_trend['value_bn'] = ext_trend['value'] / 1e9

fig = px.line(
    ext_trend, x='year', y='value_bn', color='externality',
    title='<b>📈 Externality Costs Trend — Global (2015–2030)</b>',
    labels={'value_bn': 'USD Billions', 'year': 'Year', 'externality': 'Externality'},
    markers=True, template=PLOTLY_TEMPLATE
)
fig.update_layout(height=500, width=900, xaxis=dict(dtick=1))
fig.show()

<a id='10'></a>
## 10. 🌏 Regional & Continental Analysis

In [ ]:
# ============================================================
# REGIONAL TOTAL SUBSIDIES — GROUPED BAR
# ============================================================
regional_total = df_usd[
    (df_usd['indicator'] == total_indicator)
].groupby(['year', 'region'])['value'].sum().reset_index()
regional_total['value_bn'] = regional_total['value'] / 1e9

fig = px.bar(
    regional_total, x='year', y='value_bn', color='region',
    title='<b>🌏 Regional Fossil Fuel Subsidies — Stacked (USD Billions)</b>',
    labels={'value_bn': 'USD Billions', 'year': 'Year', 'region': 'Region'},
    barmode='stack',
    color_discrete_sequence=COLOR_PALETTE,
    template=PLOTLY_TEMPLATE
)
fig.update_layout(height=550, width=1000, xaxis=dict(dtick=1))
fig.show()

In [ ]:
# ============================================================
# REGIONAL AVERAGE % GDP — HEATMAP
# ============================================================
regional_gdp = df_gdp[
    df_gdp['indicator'] == total_indicator
].groupby(['region', 'year'])['value'].mean().reset_index()

heatmap_data = regional_gdp.pivot(index='region', columns='year', values='value')

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(
    heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Average % of GDP'}
)
ax.set_title('\n🗺️ Average Fossil Fuel Subsidies (% of GDP) by Region & Year\n', fontsize=16, fontweight='bold')
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BOX PLOT — DISTRIBUTION BY REGION (2025)
# ============================================================
box_data = df_gdp[
    (df_gdp['indicator'] == total_indicator) & (df_gdp['year'] == 2025)
]

fig = px.box(
    box_data, x='region', y='value', color='region',
    title='<b>📦 Distribution of Subsidies (% GDP) by Region — 2025</b>',
    labels={'value': 'Subsidies (% of GDP)', 'region': ''},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=COLOR_PALETTE,
    points='all'
)
fig.update_layout(height=550, width=1000, showlegend=False, xaxis_tickangle=-30)
fig.show()

<a id='11'></a>
## 11. 🔍 Correlation & Distribution Analysis

In [ ]:
# ============================================================
# DISTRIBUTION OF SUBSIDIES (% GDP) — HISTOGRAM + KDE
# ============================================================
hist_data = df_gdp[
    (df_gdp['indicator'] == total_indicator) & (df_gdp['year'] == 2025)
]['value']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(hist_data, bins=30, color='#E74C3C', edgecolor='white', alpha=0.8)
axes[0].axvline(hist_data.mean(), color='#2C3E50', linestyle='--', linewidth=2, label=f'Mean: {hist_data.mean():.1f}%')
axes[0].axvline(hist_data.median(), color='#3498DB', linestyle='--', linewidth=2, label=f'Median: {hist_data.median():.1f}%')
axes[0].set_title('Distribution of Subsidies (% GDP) — 2025', fontweight='bold')
axes[0].set_xlabel('Subsidies (% of GDP)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# KDE by Region
for region in sorted(box_data['region'].unique()):
    region_vals = box_data[box_data['region'] == region]['value']
    if len(region_vals) > 2:
        region_vals.plot.kde(ax=axes[1], label=region, linewidth=2)
axes[1].set_title('KDE by Region — 2025', fontweight='bold')
axes[1].set_xlabel('Subsidies (% of GDP)')
axes[1].set_xlim(-5, 50)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CORRELATION MATRIX — FUEL TYPES (% GDP)
# ============================================================
fuel_gdp_indicators = [
    'Fossil Fuel Subsidies - Total Implicit and Explicit - Coal',
    'Fossil Fuel Subsidies - Total Implicit and Explicit -  Natural Gas',
    'Fossil Fuel Subsidies - Total Implicit and Explicit - Petroleum',
    'Fossil Fuel Subsidies - Total Implicit and Explicit - Electricity'
]

corr_data = df_gdp[
    (df_gdp['indicator'].isin(fuel_gdp_indicators)) & (df_gdp['year'] == 2025)
].pivot_table(index='country', columns='indicator', values='value')
corr_data.columns = ['Coal', 'Electricity', 'Natural Gas', 'Petroleum']

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_data.corr(), dtype=bool))
sns.heatmap(
    corr_data.corr(), annot=True, fmt='.2f', cmap='coolwarm',
    mask=mask, linewidths=1, linecolor='white', ax=ax,
    vmin=-1, vmax=1, center=0, square=True
)
ax.set_title('\nCorrelation Between Fuel Type Subsidies (% GDP, 2025)\n', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SCATTER — EXPLICIT vs IMPLICIT (% GDP) by Country, 2025
# ============================================================
scatter_exp = df_gdp[
    (df_gdp['indicator'] == 'Explicit Fossil Fuel Subsidies - Total') & (df_gdp['year'] == 2025)
][['country', 'country_code', 'value', 'region']].rename(columns={'value': 'explicit_pct'})

scatter_imp = df_gdp[
    (df_gdp['indicator'] == 'Implicit Fossil Fuel Subsidies - Total') & (df_gdp['year'] == 2025)
][['country', 'value']].rename(columns={'value': 'implicit_pct'})

scatter_df = scatter_exp.merge(scatter_imp, on='country')

fig = px.scatter(
    scatter_df, x='explicit_pct', y='implicit_pct',
    color='region', hover_name='country',
    size=scatter_df['explicit_pct'] + scatter_df['implicit_pct'],
    title='<b>🔎 Explicit vs Implicit Subsidies (% GDP) — 2025</b>',
    labels={'explicit_pct': 'Explicit Subsidies (% GDP)', 'implicit_pct': 'Implicit Subsidies (% GDP)'},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=COLOR_PALETTE
)
fig.update_layout(height=600, width=900)
fig.show()

In [ ]:
# ============================================================
# ANIMATED CHOROPLETH — TOTAL SUBSIDIES % GDP OVER TIME
# ============================================================
anim_data = df_gdp[
    df_gdp['indicator'] == total_indicator
].copy()

fig = px.choropleth(
    anim_data,
    locations='country_code',
    color='value',
    hover_name='country',
    animation_frame='year',
    color_continuous_scale='YlOrRd',
    range_color=[0, anim_data['value'].quantile(0.95)],
    title='<b>🌍 Fossil Fuel Subsidies (% GDP) — Animated Timeline</b>',
    labels={'value': '% of GDP'},
    template=PLOTLY_TEMPLATE
)
fig.update_layout(
    height=550, width=1000,
    geo=dict(showframe=False, showcoastlines=True),
    sliders=[dict(currentvalue=dict(prefix='Year: '))]
)
fig.show()

In [ ]:
# ============================================================
# TOP 10 COUNTRIES — SMALL MULTIPLES (Trend Over Time)
# ============================================================
top10_countries = df_usd[
    (df_usd['indicator'] == total_indicator) & (df_usd['year'] == 2025)
].nlargest(10, 'value')['country'].tolist()

top10_trend = df_gdp[
    (df_gdp['indicator'] == total_indicator) &
    (df_gdp['country'].isin(top10_countries))
]

fig = px.line(
    top10_trend, x='year', y='value', color='country',
    title='<b>📈 Top 10 Subsidizing Countries — Trend (% of GDP)</b>',
    labels={'value': '% of GDP', 'year': 'Year', 'country': 'Country'},
    markers=True, template=PLOTLY_TEMPLATE
)
fig.update_layout(height=550, width=1000, xaxis=dict(dtick=1))
fig.show()

<a id='12'></a>
## 12. 💡 Key Insights & Conclusions

### Summary of Findings

| Insight | Details |
|---|---|
| **Global Scale** | Fossil fuel subsidies amount to trillions of USD annually |
| **Implicit Dominance** | Implicit subsidies (externality costs) vastly outweigh explicit (direct) subsidies |
| **Top Subsidizers** | China, United States, Russia, India form the bulk in absolute terms |
| **GDP Impact** | Oil-exporting nations show the highest subsidies relative to GDP |
| **Growth Trend** | Global subsidies show a rising trend through 2030 projections |
| **Air Pollution** | Local air pollution is often the largest implicit cost component |
| **Coal Dominance** | Coal drives a disproportionate share of implicit costs due to health externalities |
| **Regional Variation** | Middle East & East Asia lead in absolute terms; MENA leads in % GDP |

### Policy Implications
- **Getting prices right** — Reforming fossil fuel pricing could reduce global CO₂ emissions by ~30%
- **Revenue potential** — Eliminating subsidies could raise revenues equivalent to ~3.8% of global GDP
- **Health benefits** — Addressing air pollution subsidies alone could prevent millions of premature deaths
- **Climate targets** — Subsidy reform is essential for Paris Agreement commitments

---

### References
1. IMF Climate Change Indicators — [climatedata.imf.org](https://climatedata.imf.org/)
2. Parry, I. et al. (2021). *Still Not Getting Energy Prices Right*. IMF Working Paper.
3. Black, S. et al. (2023). *IMF Fossil Fuel Subsidies Data: 2023 Update*.

---

<div style='text-align:center; padding:20px; background:#f8f9fa; border-radius:10px; margin-top:20px'>
    <h3>👍 If you found this analysis useful, please <b>upvote</b> the dataset on Kaggle!</h3>
    <p>Made with ❤️ by <b>Khurram Shahzad</b></p>
    <p>Mentor with ❤️ by <b>Dr. Aammar Tufail</b></p>
</div>